# CAT — Candle Analysis Tool
## BTC/USDT 1-Minute Ahead Price Prediction

**What this notebook does:**
1. Pulls 1m OHLCV data from Binance (no API key needed)
2. Builds the Order Book Imbalance feature
3. Constructs sliding-window tensors (150 candles → predict candle 151)
4. Trains a Transformer Encoder regression model (~33M params)
5. Evaluates performance on out-of-sample data with charts

**Prediction target:** Close price of the **next 1-minute candle** (T+1).

**Model:** Transformer Encoder (encoder-only, NOT GPT/decoder) with MSE regression loss.

---
## 0. Install dependencies

In [ ]:
# Run once — skip if already installed
import subprocess, sys
pkgs = [
    "ccxt", "tensorflow", "scikit-learn",
    "pandas", "numpy", "matplotlib", "tqdm", "joblib"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("All packages ready.")

---
## 1. Configuration

In [ ]:
# ── Global config ─────────────────────────────────────────────────────────────

EXCHANGE        = "bybit"
SYMBOL          = "BTC/USDT"
TIMEFRAME       = "1m"

N_CANDLES       = 51_773
OB_DEPTH        = 25
WINDOW          = 150
VAL_SPLIT       = 0.15

D_MODEL         = 512
NUM_HEADS       = 8
FF_DIM          = 1024
N_BLOCKS        = 6
DENSE1          = 1024
DENSE2          = 512
DROPOUT         = 0.2

EPOCHS          = 80
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-4

# Classification threshold — only signal when model is confident
# 0.0 = signal on every candle, 0.2 = only when prob >70% or <30%
CONFIDENCE_THRESHOLD = 0.15   # issue signal when |prob - 0.5| > this

print("Config ready.")
print(f"  Task   : BINARY CLASSIFICATION — predict UP (1) or DOWN (0)")
print(f"  Loss   : binary_crossentropy  (optimises direction directly)")
print(f"  Signal : LONG if prob > {0.5+CONFIDENCE_THRESHOLD:.2f}  |  SHORT if prob < {0.5-CONFIDENCE_THRESHOLD:.2f}")
print(f"  Data   : {N_CANDLES:,} candles  ({N_CANDLES/60/24:.1f} days)")

---
## 2. Data Collection — 1m OHLCV from Binance

In [ ]:
import requests, zipfile, io
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from tqdm.notebook import tqdm

def fetch_ohlcv_binance_s3(symbol_raw, n_candles):
    BASE = "https://data.binance.vision/data/spot/monthly/klines"
    sym  = symbol_raw.replace("/", "")

    months_needed = (n_candles // 43_200) + 3
    today   = datetime.now(timezone.utc)
    targets = [(today - timedelta(days=30*i)).strftime("%Y-%m")
               for i in range(months_needed)][::-1]

    frames = []
    for ym in tqdm(targets, desc="Downloading monthly files"):
        url  = f"{BASE}/{sym}/1m/{sym}-1m-{ym}.zip"
        resp = requests.get(url, timeout=30)
        if resp.status_code == 404:
            continue
        resp.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
            with z.open(z.namelist()[0]) as f:
                raw = pd.read_csv(f, header=None, dtype=str)

        chunk = raw.iloc[:, :6].copy()
        chunk.columns = ["timestamp","open","high","low","close","volume"]

        # Drop non-numeric rows (embedded headers)
        chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="coerce")
        chunk = chunk.dropna(subset=["timestamp"]).copy()
        chunk["timestamp"] = chunk["timestamp"].astype("int64")

        for col in ["open","high","low","close","volume"]:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
        chunk.dropna(inplace=True)
        frames.append(chunk)

    df = pd.concat(frames, ignore_index=True)

    # Binance S3 timestamps are in MICROSECONDS (16-digit), not milliseconds
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="us", utc=True)
    df = df.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)
    df = df.tail(n_candles).reset_index(drop=True)
    return df

df_ohlcv = fetch_ohlcv_binance_s3(SYMBOL, N_CANDLES)
print(f"\nFetched {len(df_ohlcv):,} candles")
print(f"Date range: {df_ohlcv['timestamp'].iloc[0]}  →  {df_ohlcv['timestamp'].iloc[-1]}")
df_ohlcv.tail(5)

### 2a. Visualise raw price

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

# Close price
axes[0].plot(df_ohlcv["timestamp"], df_ohlcv["close"], linewidth=0.8, color="#1f77b4")
axes[0].set_title(f"{SYMBOL} {TIMEFRAME} Close Price", fontsize=13)
axes[0].set_ylabel("Price (USD)")
axes[0].grid(alpha=0.3)

# Volume
axes[1].bar(df_ohlcv["timestamp"], df_ohlcv["volume"],
            width=pd.Timedelta(seconds=50), color="#ff7f0e", alpha=0.6)
axes[1].set_title("Volume", fontsize=13)
axes[1].set_ylabel("Volume (BTC)")
axes[1].set_xlabel("Time (UTC)")
axes[1].grid(alpha=0.3)

axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 3. Order Book Imbalance Feature

$$OBI = \frac{\sum BidVol - \sum AskVol}{\sum BidVol + \sum AskVol} \in [-1, +1]$$

**+1** = all pressure on the buy side → bullish  
**−1** = all pressure on the sell side → bearish

For historical data we approximate OBI from candle shape (candle body / candle range). This is a proxy; in live mode the real L2 book is used.

In [ ]:
# ── FIX 2: add technical indicator features for direction signal ──────────────
# Features added:
#   ob_imbalance  — candle body pressure proxy
#   rsi_14        — momentum: >70 overbought, <30 oversold
#   ema_diff      — EMA9 - EMA21: trend direction
#   vol_ratio     — volume / 20-bar rolling mean: volume spike detector
#   price_accel   — (close[t] - close[t-1]) - (close[t-1] - close[t-2]): acceleration

df = df_ohlcv.copy()

# 1. OBI proxy
candle_range = (df["high"] - df["low"]).replace(0, np.nan)
df["ob_imbalance"] = ((df["close"] - df["open"]) / candle_range).clip(-1, 1).fillna(0)

# 2. RSI-14
delta    = df["close"].diff()
gain     = delta.clip(lower=0).rolling(14).mean()
loss     = (-delta.clip(upper=0)).rolling(14).mean()
rs       = gain / loss.replace(0, np.nan)
df["rsi_14"] = (100 - 100 / (1 + rs)).fillna(50) / 100   # scaled 0→1

# 3. EMA diff (EMA9 - EMA21), normalised by price
ema9     = df["close"].ewm(span=9,  adjust=False).mean()
ema21    = df["close"].ewm(span=21, adjust=False).mean()
df["ema_diff"] = ((ema9 - ema21) / df["close"]).fillna(0)

# 4. Volume ratio (vol / 20-bar mean)
vol_mean = df["volume"].rolling(20).mean().replace(0, np.nan)
df["vol_ratio"] = (df["volume"] / vol_mean).fillna(1).clip(0, 10) / 10   # scaled 0→1

# 5. Price acceleration
ret      = df["close"].diff()
df["price_accel"] = ret.diff().fillna(0) / df["close"]

# Drop NaN rows from rolling windows (first ~21 rows)
df_ohlcv = df.dropna().reset_index(drop=True)

print(f"Rows after feature engineering: {len(df_ohlcv):,}")
print(f"New features: ob_imbalance, rsi_14, ema_diff, vol_ratio, price_accel")
print(df_ohlcv[["timestamp","close","ob_imbalance","rsi_14","ema_diff","vol_ratio","price_accel"]].tail(3))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))
ax.fill_between(df_ohlcv["timestamp"], df_ohlcv["ob_imbalance"],
                where=df_ohlcv["ob_imbalance"] >= 0,
                color="green", alpha=0.5, label="Bid pressure")
ax.fill_between(df_ohlcv["timestamp"], df_ohlcv["ob_imbalance"],
                where=df_ohlcv["ob_imbalance"] < 0,
                color="red", alpha=0.5, label="Ask pressure")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Order Book Imbalance (proxy from candle body)", fontsize=13)
ax.set_ylabel("OBI")
ax.set_xlabel("Time (UTC)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

---
## 4. Build Tensors

**Input X:** shape `(N, 150, 6)` — 150 candles × 6 features `[open, high, low, close, volume, ob_imbalance]`  
**Label y:** shape `(N, 1)` — the **close price of candle 151** (the very next minute)  

Normalisation is fit **only on the training split** to prevent data leakage.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib, os

FEATURES = [
    "open", "high", "low", "close", "volume",
    "ob_imbalance", "rsi_14", "ema_diff", "vol_ratio", "price_accel"
]

values = df_ohlcv[FEATURES].values.astype(np.float64)

split      = int(len(values) * (1 - VAL_SPLIT))
train_vals = values[:split]
val_vals   = values[split:]

print(f"Total rows : {len(values):,}")
print(f"Train rows : {len(train_vals):,}  ({100*(1-VAL_SPLIT):.0f}%)")
print(f"Val rows   : {len(val_vals):,}   ({100*VAL_SPLIT:.0f}%)")
print(f"Features   : {FEATURES}")

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_vals)
train_scaled = scaler.transform(train_vals)
val_scaled   = scaler.transform(val_vals)

def build_windows(arr, window):
    n = len(arr) - window
    X = np.lib.stride_tricks.sliding_window_view(arr, (window, arr.shape[1]))
    X = X[:n].reshape(n, window, arr.shape[1]).astype(np.float32)

    # y = 1 if next candle closes HIGHER than current, else 0
    next_close    = arr[window    : window + n, 3]
    current_close = arr[window - 1: window + n - 1, 3]
    y = (next_close > current_close).astype(np.float32).reshape(-1, 1)
    return X, y

X_train, y_train = build_windows(train_scaled, WINDOW)
X_val,   y_val   = build_windows(val_scaled,   WINDOW)

up_pct = y_train.mean() * 100
print(f"\nX_train : {X_train.shape}")
print(f"y_train : {y_train.shape}  ← 1=UP  0=DOWN")
print(f"Class balance: {up_pct:.1f}% UP  /  {100-up_pct:.1f}% DOWN  (50/50 = balanced)")

---
## 5. Model Architecture — Transformer Encoder

```
Input (150, 6)
  → Dense projection → d_model
  → Sinusoidal Positional Encoding
  → N × [MultiHeadAttention → LayerNorm → FFN → LayerNorm]
  → GlobalAveragePooling1D
  → Dense(1024, relu) → Dense(512, relu) → Dense(1)
```

> This is an **encoder-only** Transformer (like BERT), not a GPT decoder.  
> The model reads the full 150-candle window simultaneously via self-attention,  
> then regresses onto a single price value.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"GPU detected: {gpus[0].name}")
else:
    print("No GPU detected — training on CPU (will be slower)")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"GPU detected: {gpus[0].name}")
else:
    print("No GPU — training on CPU")

class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        positions = tf.cast(tf.range(max_len)[:, None], tf.float32)
        dims      = tf.cast(tf.range(d_model)[None, :], tf.float32)
        angles    = positions / tf.pow(10000.0, (2*(dims//2)) / tf.cast(d_model, tf.float32))
        sin_enc   = tf.math.sin(angles[:, 0::2])
        cos_enc   = tf.math.cos(angles[:, 1::2])
        enc       = tf.reshape(tf.stack([sin_enc, cos_enc], axis=-1), [max_len, -1])
        self.encoding = tf.cast(enc[None, :, :d_model], tf.float32)

    def call(self, x):
        return x + self.encoding[:, :tf.shape(x)[1], :]

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = layers.MultiHeadAttention(num_heads=num_heads,
                                               key_dim=d_model // num_heads,
                                               dropout=dropout)
        self.ffn   = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.attn(x, x, training=training)
        x = self.norm1(x + self.drop1(attn_out, training=training))
        x = self.norm2(x + self.drop2(self.ffn(x), training=training))
        return x

def build_cat_model(window, n_features, d_model, num_heads, ff_dim,
                    n_blocks, dense1, dense2, dropout):
    inputs = keras.Input(shape=(window, n_features), name="candles")
    x = layers.Dense(d_model, name="input_projection")(inputs)
    x = PositionalEncoding(max_len=window, d_model=d_model, name="pos_enc")(x)
    for i in range(n_blocks):
        x = TransformerBlock(d_model, num_heads, ff_dim, dropout,
                             name=f"transformer_block_{i}")(x)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(dense1, activation="relu", name="dense_1")(x)
    x = layers.Dropout(dropout, name="drop_1")(x)
    x = layers.Dense(dense2, activation="relu", name="dense_2")(x)
    x = layers.Dropout(dropout, name="drop_2")(x)
    # sigmoid output → probability of UP (1=up, 0=down)
    output = layers.Dense(1, activation="sigmoid", name="direction_output")(x)
    return keras.Model(inputs=inputs, outputs=output, name="CAT_Classifier")

model = build_cat_model(
    window=WINDOW, n_features=len(FEATURES),
    d_model=D_MODEL, num_heads=NUM_HEADS, ff_dim=FF_DIM,
    n_blocks=N_BLOCKS, dense1=DENSE1, dense2=DENSE2, dropout=DROPOUT
)

# binary_crossentropy directly optimises for correct direction
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]      # accuracy = direction accuracy
)

total_params = model.count_params()
model.summary()
print(f"\nTotal parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")
print("Output: sigmoid probability — >0.5 = UP prediction, <0.5 = DOWN prediction")

---
## 6. Training

In [ ]:
os.makedirs("saved_model", exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="saved_model/cat_model_nb.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

joblib.dump(scaler, "saved_model/scaler_nb.joblib")
print("Scaler saved → saved_model/scaler_nb.joblib")

### 6a. Training curves

In [ ]:
hist = history.history
epochs_ran = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE loss
axes[0].plot(epochs_ran, hist["loss"],     label="Train MSE",      linewidth=2)
axes[0].plot(epochs_ran, hist["val_loss"], label="Val MSE",  linestyle="--", linewidth=2)
axes[0].set_title("MSE Loss", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE (scaled)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE
axes[1].plot(epochs_ran, hist["mae"],     label="Train MAE",      linewidth=2, color="green")
axes[1].plot(epochs_ran, hist["val_mae"], label="Val MAE",  linestyle="--", linewidth=2, color="orange")
axes[1].set_title("Mean Absolute Error", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (scaled)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("CAT Transformer — Training History", fontsize=15)
plt.tight_layout()
plt.savefig("saved_model/training_curves.png", dpi=150)
plt.show()
print("Saved → saved_model/training_curves.png")

---
## 7. Evaluation — Out-of-Sample Performance

The validation set is the **last 15% of data** — data the model has **never seen**.

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

# Raw probabilities from sigmoid
y_prob = model.predict(X_val, verbose=0).flatten()   # 0.0 → 1.0

# Hard prediction: >0.5 = UP (1), <=0.5 = DOWN (0)
y_pred_dir = (y_prob > 0.5).astype(int)
y_true_dir = y_val.flatten().astype(int)

# Direction accuracy
dir_acc_all = (y_pred_dir == y_true_dir).mean() * 100

# Confidence-filtered signals
up_mask   = y_prob > (0.5 + CONFIDENCE_THRESHOLD)   # confident LONG
down_mask = y_prob < (0.5 - CONFIDENCE_THRESHOLD)   # confident SHORT
hold_mask = ~(up_mask | down_mask)

signals = np.where(up_mask, 1, np.where(down_mask, -1, 0))
signal_mask = signals != 0

hits    = (signals[signal_mask] == (y_true_dir[signal_mask] * 2 - 1)).sum()
dir_acc = hits / signal_mask.sum() * 100 if signal_mask.sum() > 0 else 0.0

# Probability distribution
print("═" * 55)
print("  Out-of-Sample Classification Results")
print("═" * 55)
print(f"  Loss (binary_crossentropy) : {val_loss:.6f}")
print(f"  Direction accuracy (all)   : {val_acc*100:.2f}%")
print(f"  Direction accuracy (check) : {dir_acc_all:.2f}%")
print(f"\n  Probability distribution:")
print(f"    mean  : {y_prob.mean():.4f}  (0.5 = perfectly uncertain)")
print(f"    std   : {y_prob.std():.4f}  (higher = more decisive)")
print(f"    >0.65 : {(y_prob>0.65).mean()*100:.1f}%  (high-confidence UP)")
print(f"    <0.35 : {(y_prob<0.35).mean()*100:.1f}%  (high-confidence DOWN)")
print("═" * 55)

In [ ]:
# ── Predicted vs Actual chart ─────────────────────────────────────────────────
PLOT_N = min(500, len(y_true_usd))  # show last 500 val samples for clarity

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# Chart 1: Overlaid prices
idx = np.arange(PLOT_N)
axes[0].plot(idx, y_true_usd[-PLOT_N:], label="Actual close (T+1)",    linewidth=1.2, color="#1f77b4")
axes[0].plot(idx, y_pred_usd[-PLOT_N:], label="Predicted close (T+1)", linewidth=1.0,
             color="#ff7f0e", linestyle="--", alpha=0.85)
axes[0].set_title(f"CAT — Actual vs Predicted (last {PLOT_N} validation samples)", fontsize=13)
axes[0].set_ylabel("BTC Price (USD)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Chart 2: Residuals (prediction error)
residuals = y_pred_usd[-PLOT_N:] - y_true_usd[-PLOT_N:]
axes[1].bar(idx, residuals,
            color=np.where(residuals >= 0, "#2ca02c", "#d62728"),
            alpha=0.6, width=1.0)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].axhline( mae_usd, color="orange", linestyle="--", linewidth=1.2, label=f"+MAE ${mae_usd:.0f}")
axes[1].axhline(-mae_usd, color="orange", linestyle="--", linewidth=1.2, label=f"−MAE ${mae_usd:.0f}")
axes[1].set_title("Prediction Error (Predicted − Actual)", fontsize=13)
axes[1].set_ylabel("Error (USD)")
axes[1].set_xlabel("Validation sample index")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("saved_model/prediction_vs_actual.png", dpi=150)
plt.show()
print("Saved → saved_model/prediction_vs_actual.png")

In [ ]:
# ── Scatter: predicted vs actual ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true_usd, y_pred_usd, alpha=0.2, s=4, color="steelblue")
lo = min(y_true_usd.min(), y_pred_usd.min())
hi = max(y_true_usd.max(), y_pred_usd.max())
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Price (USD)")
ax.set_ylabel("Predicted Price (USD)")
ax.set_title("Scatter: Predicted vs Actual", fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("saved_model/scatter_pred_actual.png", dpi=150)
plt.show()

In [ ]:
# ── Error distribution histogram ──────────────────────────────────────────────
all_residuals = y_pred_usd - y_true_usd

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_residuals, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(0,            color="black",  linewidth=1.5, label="Zero error")
ax.axvline( mae_usd,     color="orange", linewidth=1.5, linestyle="--", label=f"+MAE ${mae_usd:.0f}")
ax.axvline(-mae_usd,     color="orange", linewidth=1.5, linestyle="--", label=f"−MAE ${mae_usd:.0f}")
ax.set_title("Prediction Error Distribution (full validation set)", fontsize=13)
ax.set_xlabel("Error (USD)")
ax.set_ylabel("Count")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("saved_model/error_distribution.png", dpi=150)
plt.show()

print(f"Errors within ±$100 : {(np.abs(all_residuals) <= 100).mean()*100:.1f}% of predictions")
print(f"Errors within ±$500 : {(np.abs(all_residuals) <= 500).mean()*100:.1f}% of predictions")

---
## 8. Signal Generation Demo

Simulate what the live agent would see: take the last 150 candles and produce a trading signal.

In [ ]:
# ── Live signal demo ─────────────────────────────────────────────────────────
last_window = X_val[-1]
prob = float(model.predict(last_window[np.newaxis], verbose=0)[0, 0])

signal     = "LONG"  if prob > (0.5 + CONFIDENCE_THRESHOLD) else \
             "SHORT" if prob < (0.5 - CONFIDENCE_THRESHOLD) else "HOLD"
confidence = abs(prob - 0.5) / 0.5 * 100   # 0% = no idea, 100% = certain

# Recover current price
dummy = np.zeros((1, len(FEATURES)))
dummy[0, 3] = last_window[-1, 3]
current_price = scaler.inverse_transform(dummy)[0, 3]

print("═" * 50)
print("  LIVE SIGNAL DEMO")
print("═" * 50)
print(f"  Current price : ${current_price:,.2f}")
print(f"  UP probability: {prob:.4f}  ({prob*100:.1f}%)")
print(f"  Signal        : {signal}")
print(f"  Confidence    : {confidence:.1f}%")
print(f"  Threshold     : >{0.5+CONFIDENCE_THRESHOLD:.2f} = LONG  |  <{0.5-CONFIDENCE_THRESHOLD:.2f} = SHORT")
print("═" * 50)

In [ ]:
# ── Signal backtest ──────────────────────────────────────────────────────────
print(f"Total val candles : {len(signals):,}")
print(f"Signals issued    : {signal_mask.sum():,}  ({signal_mask.mean()*100:.1f}%)")
print(f"  LONG  : {(signals==1).sum():,}")
print(f"  SHORT : {(signals==-1).sum():,}")
print(f"  HOLD  : {(signals==0).sum():,}")
print(f"\nDirection accuracy (all candles) : {dir_acc_all:.2f}%")
print(f"Direction accuracy (on signals)  : {dir_acc:.2f}%")
print(f"\n  Confidence={CONFIDENCE_THRESHOLD} means signal only when prob >{0.5+CONFIDENCE_THRESHOLD:.2f} or <{0.5-CONFIDENCE_THRESHOLD:.2f}")
print(f"  Lower CONFIDENCE_THRESHOLD → more signals, possibly lower accuracy")
print(f"  Higher CONFIDENCE_THRESHOLD → fewer signals, possibly higher accuracy")

---
## 9. Summary

| Item | Value |
|---|---|
| **Prediction target** | Close price of the **next 1-minute candle** (T+1) |
| **Model type** | Transformer Encoder (encoder-only, regression) |
| **Input** | 150 × 6 matrix `[O, H, L, C, V, OBI]` |
| **Output** | Single float — predicted close price |
| **Loss** | MSE (Mean Squared Error) |
| **Key feature** | Order Book Imbalance captures bid/ask pressure |
| **Validation** | Last 15% of data (temporal, no leakage) |
| **Data** | **51,773 rows** · 35 days · ~17 MB (CAT 1.3 exact spec) |
| **Parameters** | **~33M** (d_model=256, 6 blocks, 8 heads, Dense 1024→512→1) |

In [ ]:
import numpy as np

lines = []
lines.append("=" * 60)
lines.append("  CAT MODEL EVALUATION REPORT")
lines.append("=" * 60)

lines.append("\n[CONFIG]")
lines.append(f"  N_CANDLES              : {N_CANDLES:,}")
lines.append(f"  WINDOW                 : {WINDOW}")
lines.append(f"  D_MODEL                : {D_MODEL}")
lines.append(f"  N_BLOCKS               : {N_BLOCKS}")
lines.append(f"  NUM_HEADS              : {NUM_HEADS}")
lines.append(f"  DENSE1 / DENSE2        : {DENSE1} / {DENSE2}")
lines.append(f"  EPOCHS ran             : {len(history.history['loss'])}")
lines.append(f"  BATCH_SIZE             : {BATCH_SIZE}")
lines.append(f"  Total params           : {model.count_params():,}")
lines.append(f"  Task                   : Binary classification (UP/DOWN)")
lines.append(f"  CONFIDENCE_THRESHOLD   : {CONFIDENCE_THRESHOLD}")

lines.append("\n[DATA SPLIT]")
lines.append(f"  Train samples  : {len(X_train):,}")
lines.append(f"  Val samples    : {len(X_val):,}")
lines.append(f"  Features       : {FEATURES}")
lines.append(f"  Class balance  : {y_train.mean()*100:.1f}% UP  /  {100-y_train.mean()*100:.1f}% DOWN")

lines.append("\n[METRICS — out-of-sample]")
lines.append(f"  Loss (binary_crossentropy)       : {val_loss:.6f}")
lines.append(f"  Direction accuracy (all)         : {dir_acc_all:.2f}%")
lines.append(f"  Direction accuracy (on signals)  : {dir_acc:.2f}%")

lines.append("\n[PROBABILITY DISTRIBUTION]")
lines.append(f"  mean  : {y_prob.mean():.4f}")
lines.append(f"  std   : {y_prob.std():.4f}")
lines.append(f"  >0.65 : {(y_prob>0.65).mean()*100:.1f}%  (high-confidence UP)")
lines.append(f"  <0.35 : {(y_prob<0.35).mean()*100:.1f}%  (high-confidence DOWN)")

lines.append(f"\n[SIGNAL BACKTEST]  confidence_threshold={CONFIDENCE_THRESHOLD}")
lines.append(f"  Total val candles : {len(signals):,}")
lines.append(f"  Signals issued    : {signal_mask.sum():,}  ({signal_mask.mean()*100:.1f}%)")
lines.append(f"  LONG              : {(signals==1).sum():,}")
lines.append(f"  SHORT             : {(signals==-1).sum():,}")
lines.append(f"  HOLD              : {(signals==0).sum():,}")
lines.append(f"  Direction acc (on signals) : {dir_acc:.2f}%")

train_loss = history.history['loss']
val_loss_h = history.history['val_loss']
best_epoch = int(np.argmin(val_loss_h)) + 1
lines.append("\n[TRAINING CURVE]")
lines.append(f"  Best epoch       : {best_epoch}")
lines.append(f"  Best val loss    : {min(val_loss_h):.6f}")
lines.append(f"  Final train loss : {train_loss[-1]:.6f}")
lines.append(f"  Final val loss   : {val_loss_h[-1]:.6f}")
lines.append(f"  Overfit gap      : {val_loss_h[-1] - train_loss[-1]:+.6f}")

lines.append("\n" + "=" * 60)
report = "\n".join(lines)

with open("eval_report.txt", "w") as f:
    f.write(report)

print(report)
print("\nSaved → eval_report.txt")